# Construcción de un espacio semántico

> Ideas clave:

* Cómo representar una palabra en un espacio, donde la distancia hacía otra palabra tenga un sentido semántico o contextual, por ejemplo, la frecuencia y distancia entre palabras
* Cómo reducir el espacio semántico a vectores de máxima información con pocas dimensiones

TEXTO -> TOKENS -> INDEXARLOS -> EMBEDDING (MATRIZ DE CORRELACIÓN / TRANSFORMERS)

## Paso 1. Extraer tokens del texto

In [3]:
archivo = open("../conjuntos/quijote.txt", "r") # Cursor / Objeto al archivo

quijote = archivo.read() # Lee todo el archivo y 
                         # lo carga a memoria como un string <str> (Texto)

archivo.close() # Cerrar el archivo

In [4]:
len(quijote) # Tamaño del string - len(<str>)

1299863

In [5]:
# Autoclausura (cierra solo el archivo)
with open("../conjuntos/quijote.txt", "r") as archivo:
    quijote = archivo.read() # <str>

quijote[:100] # 100 - Caracteres | type(quitoje) <str>

'Miguel de Cervantes Saavedra\nEl ingenioso hidalgo don Quijote de la Mancha\n\nAl Duque de Béjar MARQUÉ'

> Normalización del texto

Manzana <-> manzana

<CA>manzana <-> manzana

Público <-> público

<CA>pu<AC>blico <-> pu<AC>blico

público <-> publicó

pu<AC>blico <-> publico<AC>

publico<AE> <-> publico<AA>

"Hola, ¿Cómo están todos este año?"

[h, o, l, a, <PC>, <PS>, <IA>, <CA>, c, o, <AG>, <PS>, ]

Algunas tokenizaciones:

- Por palabras y espacios (simple)
- Por letras y puntuaciones (fina)
- Por palabras y puntuaciones (estructural)
- N-Gramas de letras (3-grama combina 3 letras o puntuaciones)

Ejemplos:

* [hol, a<sp>m, und, o<em><em>] - 3-grama
* [h, o, l, a, <sp>, m, u, n, d, o, <em>, <em>] - letras y puntuaciones (fina)
* [hola, <sp>mun, do<em><em>] -  4-grama
* [hola, <sp>, mundo] - palabras y espacios
* [don, <PS>, ramon<AA>] - palabras y puntuaciones (estructural)
* [don, <PS>, ramo<AC>n] - palabras y puntuaciones (no estructural)

In [6]:
import unicodedata

marcas = {
    32: "<PS>",
    44: "<PC>",
    191: "<IA>",
    63: "<IC>",
    769: "<AC>",
    771: "<TI>",
}

# ord(caracter) -> entero ASCII (0-255)
def tokenizar(texto):
    vector = []

    texto_normalizado = unicodedata.normalize("NFD", texto)

    for caracter in texto_normalizado:
        # Determinar si es una letra mayúscula A-Z
        if ord(caracter) >= ord("A") and ord(caracter) <= ord("Z"):
            vector.append("<MA>")
            vector.append(caracter.lower())
            continue
        # Determinar si es una letra minúscula a-z
        if ord(caracter) >= ord("a") and ord(caracter) <= ord("z"):
            vector.append(caracter.lower())
            continue
        # Determinar si es una marca de puntuación <PS>, <PC>, <IA>, <IC>, <AC>
        if ord(caracter) in marcas:
            marca = marcas[ord(caracter)]
            vector.append(marca)
            continue
        print(caracter, ord(caracter))
        vector.append("<ND>")

    # vector_a = [ord(caracter) for caracter in texto]
    # texto_normalizado = unicodedata.normalize("NFD", texto)
    # vector_b = [ord(caracter) for caracter in texto_normalizado]
    # vector_c = [chr(ord(caracter)) for caracter in texto_normalizado]
    # print(vector_a)
    # print(vector_b)
    # print(vector_c)
    return vector

tokenizar("Hola, ¿Cómo están todos este año?")

['<MA>',
 'h',
 'o',
 'l',
 'a',
 '<PC>',
 '<PS>',
 '<IA>',
 '<MA>',
 'c',
 'o',
 '<AC>',
 'm',
 'o',
 '<PS>',
 'e',
 's',
 't',
 'a',
 '<AC>',
 'n',
 '<PS>',
 't',
 'o',
 'd',
 'o',
 's',
 '<PS>',
 'e',
 's',
 't',
 'e',
 '<PS>',
 'a',
 'n',
 '<TI>',
 'o',
 '<IC>']

In [7]:
# ó -> [o, ´] -> [111, 769] -> o+Ux301
hex(769) # U

'0x301'

In [8]:
marcas = {
    32: "<PS>",
    44: "<PC>",
    191: "<IA>",
    63: "<IC>",
    769: "<AC>",
    771: "<TI>",
}

def tokenizar(texto):
    matriz = []

    texto_normalizado = unicodedata.normalize("NFD", texto)

    for palabra in texto_normalizado.split(" "): # Corte por espacios forma palabras
        vector = []
        for caracter in palabra:
            # Determinar si el carecter vacíos o blancos es menor a un espacio 32
            if ord(caracter) <= ord(" "):
                vector.append("<PS>")
                continue
            # Determinar si es una letra mayúscula A-Z
            if ord(caracter) >= ord("A") and ord(caracter) <= ord("Z"):
                vector.append("<MA>")
                vector.append(caracter.lower())
                continue
            # Determinar si es una letra minúscula a-z
            if ord(caracter) >= ord("a") and ord(caracter) <= ord("z"):
                vector.append(caracter.lower())
                continue
            # Determinar si es una marca de puntuación <PS>, <PC>, <IA>, <IC>, <AC>
            if ord(caracter) in marcas:
                marca = marcas[ord(caracter)]
                vector.append(marca)
                continue
            print(caracter, ord(caracter))
            vector.append("<ND>")
        matriz.append(vector)

    return matriz

tokenizar("Hola, ¿Cómo están todos este otoño?")

[['<MA>', 'h', 'o', 'l', 'a', '<PC>'],
 ['<IA>', '<MA>', 'c', 'o', '<AC>', 'm', 'o'],
 ['e', 's', 't', 'a', '<AC>', 'n'],
 ['t', 'o', 'd', 'o', 's'],
 ['e', 's', 't', 'e'],
 ['o', 't', 'o', 'n', '<TI>', 'o', '<IC>']]

In [9]:
import re

marcas = {
    32: "<PS>",
    44: "<PC>",
    191: "<IA>",
    63: "<IC>",
    769: "<AC>",
    771: "<TI>",
}

def tokenizar(texto, puntuaciones=True):
    matriz = []

    texto_normalizado = unicodedata.normalize("NFD", texto)

    texto_normalizado = re.sub(r"\n", " ", texto_normalizado)
    texto_normalizado = re.sub(r"\t", " ", texto_normalizado)
    texto_normalizado = re.sub(r"\s+", " ", texto_normalizado)

    for palabra in texto_normalizado.split(" "): # Corte por espacios forma palabras
        vector_base = []
        vector_marcas = []
        posicion = 0
        for caracter in palabra:
            posicion = posicion + 1
            # Determinar si el carecter vacíos o blancos es menor a un espacio 32
            if ord(caracter) <= ord(" "):
                vector_marcas.append("<PS>")
            # Determinar si es una letra mayúscula A-Z
            elif ord(caracter) >= ord("A") and ord(caracter) <= ord("Z"):
                vector_marcas.append("<MA>")
                vector_marcas.append(f"<{posicion}>")
                vector_base.append(caracter.lower())
            # Determinar si es una letra minúscula a-z
            elif ord(caracter) >= ord("a") and ord(caracter) <= ord("z"):
                vector_base.append(caracter)
            # Determinar si es una marca de puntuación <PS>, <PC>, <IA>, <IC>, <AC>
            elif ord(caracter) in marcas:
                marca = marcas[ord(caracter)]
                vector_marcas.append(marca)
                vector_marcas.append(f"<{posicion}>")
            else:
                print(caracter, ord(caracter), posicion)
                vector_marcas.append("<ND>")
        # matriz.append("".join(vector_base + vector_marcas))
        if len(vector_base) > 0:
            matriz.append("".join(vector_base))
        if len(vector_marcas) > 0:
            if puntuaciones:
                # matriz.extend(vector_marcas)
                matriz.append("".join(vector_marcas))

    return matriz

tokenizar("Hola, ¿Cómo están todos este otoño?", puntuaciones=False)

['hola', 'como', 'estan', 'todos', 'este', 'otono']

In [10]:
quijote_corpus_tokens = tokenizar(quijote, puntuaciones=True)

quijote_corpus_tokens

; 59 7
. 46 9
: 58 7
. 46 11
. 46 10
. 46 9
. 46 9
« 171 1
» 187 1
. 46 6
. 46 8
. 46 8
; 59 12
. 46 10
- 45 1
. 46 7
. 46 5
; 59 14
. 46 8
- 45 1
- 45 11
. 46 6
. 46 6
: 58 5
- 45 1
. 46 8
. 46 7
. 46 9
. 46 8
- 45 1
- 45 1
- 45 7
: 58 5
- 45 1
; 59 7
̈ 776 8
. 46 12
: 58 11
. 46 5
. 46 5
. 46 7
: 58 5
« 171 1
: 58 6
» 187 8
. 46 9
: 58 10
« 171 1
» 187 1
. 46 2
: 58 9
. 46 7
. 46 5
. 46 4
: 58 7
: 58 6
« 171 1
. 46 6
. 46 7
. 46 8
» 187 9
. 46 8
: 58 9
« 171 1
; 59 9
» 187 4
. 46 4
; 59 5
; 59 9
; 59 6
; 59 6
. 46 11
. 46 8
. 46 8
; 59 12
. 46 6
. 46 7
. 46 7
; 59 6
; 59 8
. 46 6
. 46 5
. 46 14
. 46 11
. 46 13
. 46 9
; 59 5
. 46 5
. 46 10
; 59 10
. 46 11
. 46 7
. 46 5
- 45 4
- 45 5
- 45 3
. 46 4
- 45 5
- 45 4
- 45 6
. 46 7
- 45 5
- 45 4
- 45 3
: 58 4
- 45 6
. 46 7
- 45 7
- 45 5
; 59 6
- 45 8
- 45 6
- 45 8
- 45 5
. 46 6
- 45 5
- 45 5
- 45 5
. 46 6
- 45 5
- 45 5
: 58 6
« 171 1
¡ 161 2
- 45 3
- 45 6
- 45 6
! 33 1
» 187 2
. 46 3
- 45 5
- 45 6
. 46 7
- 45 4
- 45 6
- 45 3
- 45 3
- 45 4
: 5

['miguel',
 '<MA><1>',
 'de',
 'cervantes',
 '<MA><1>',
 'saavedra',
 '<MA><1>',
 'el',
 '<MA><1>',
 'ingenioso',
 'hidalgo',
 'don',
 'quijote',
 '<MA><1>',
 'de',
 'la',
 'mancha',
 '<MA><1>',
 'al',
 '<MA><1>',
 'duque',
 '<MA><1>',
 'de',
 'bejar',
 '<MA><1><AC><3>',
 'marques',
 '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><AC><7><MA><8>',
 'de',
 '<MA><1><MA><2>',
 'gibraleon',
 '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><MA><7><MA><8><AC><9><MA><10><PC><11>',
 'conde',
 '<MA><1><MA><2><MA><3><MA><4><MA><5>',
 'de',
 '<MA><1><MA><2>',
 'benalcazar',
 '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><MA><7><AC><8><MA><9><MA><10><MA><11>',
 'y',
 '<MA><1>',
 'banares',
 '<MA><1><MA><2><MA><3><TI><4><MA><5><MA><6><MA><7><MA><8><PC><9>',
 'vizconde',
 '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><MA><7><MA><8>',
 'de',
 '<MA><1><MA><2>',
 'la',
 '<MA><1><MA><2>',
 'puebla',
 '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6>',
 'de',
 '<MA><1><MA><2>',
 'alcocersenor',
 '<MA><1><MA><2><MA><3><MA><4><

In [11]:
quijote_corpus_diccionario_token_frecuencias = {}
quijote_corpus_diccionario_token_indices = {}

indice = 1
for token in quijote_corpus_tokens:
    if not token in quijote_corpus_diccionario_token_frecuencias:
        quijote_corpus_diccionario_token_frecuencias[token] = 1
        quijote_corpus_diccionario_token_indices[token] = indice
        indice += 1
    else:
        quijote_corpus_diccionario_token_frecuencias[token] += 1

quijote_corpus_diccionario_token_frecuencias, quijote_corpus_diccionario_token_indices

({'miguel': 4,
  '<MA><1>': 6304,
  'de': 11503,
  'cervantes': 3,
  'saavedra': 4,
  'el': 5971,
  'ingenioso': 12,
  'hidalgo': 34,
  'don': 1711,
  'quijote': 1444,
  'la': 6490,
  'mancha': 114,
  'al': 1148,
  'duque': 151,
  'bejar': 2,
  '<MA><1><AC><3>': 118,
  'marques': 11,
  '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><AC><7><MA><8>': 2,
  '<MA><1><MA><2>': 40,
  'gibraleon': 1,
  '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><MA><7><MA><8><AC><9><MA><10><PC><11>': 1,
  'conde': 13,
  '<MA><1><MA><2><MA><3><MA><4><MA><5>': 34,
  'benalcazar': 1,
  '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><MA><7><AC><8><MA><9><MA><10><MA><11>': 1,
  'y': 11392,
  'banares': 1,
  '<MA><1><MA><2><MA><3><TI><4><MA><5><MA><6><MA><7><MA><8><PC><9>': 1,
  'vizconde': 1,
  '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><MA><7><MA><8>': 3,
  'puebla': 1,
  '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6>': 34,
  'alcocersenor': 1,
  '<MA><1><MA><2><MA><3><MA><4><MA><5><MA><6><MA><7><PC><8><MA><9><MA><10><MA><11

In [12]:
len(quijote_corpus_tokens)

298926

In [13]:
len(quijote_corpus_diccionario_token_indices.keys())

18355

In [14]:
quijote_corpus_diccionario_token_indices["quijote"]

10

In [15]:
quijote_corpus_diccionario_token_frecuencias["quijote"]

1444

## Encontrar la distancia entre dos tokens

Sea $t_i$ el i-ésimo token, y $t_j$ el j-ésimo token, entonces, $w_{ij}$ representará la distancia entre esos dos tokens.

Para calcular $w_{ij}$ tomamos cada $t_i$ y cada t_j y anotamos la distancia inversa entre sus dos posiciones dentro del corpus, pero a mayor distancia menor relación, por lo que la suma de inversas solo aplicará a los próximos $k$ tokens y supondremos que los demás más lejanos ya afectan su contexto.

In [16]:
18355 * 18355

336906025

In [ ]:
W = {} # Matriz de correlación en forma de diccionario (matriz de diccionario x diccionario)

tokens = quijote_corpus_tokens

k = 10

for i, token_i in enumerate(tokens):
    for j in range(i + 1, min(len(tokens), i + k)):
        token_j = tokens[j]
        # print(i, token_i, j, token_j)
        d = j - i
        if not token_i in W: 
            W[token_i] = {}
        if not token_j in W[token_i]:
            W[token_i][token_j] = 0
        W[token_i][token_j] += 1 / d # PMI = ln(P(ti)/P(ti)P(tj))

    # if i > 100:
    #     break

W

{'miguel': {'<MA><1>': 5.375000000000001,
  'de': 1.5,
  'cervantes': 1.0,
  'saavedra': 0.6000000000000001,
  'el': 0.14285714285714285,
  'ingenioso': 0.1111111111111111,
  'prologo': 0.2857142857142857,
  '<MA><1><AC><4>': 0.25,
  'desocupado': 0.1111111111111111,
  'que': 0.625,
  'vendra': 0.3333333333333333,
  '<AC><7>': 0.25,
  'dice': 0.2,
  'mi': 0.16666666666666666,
  'padre': 0.14285714285714285,
  'los': 0.1111111111111111,
  'al': 0.1111111111111111},
 '<MA><1>': {'de': 734.4198412698382,
  'cervantes': 1.777777777777778,
  '<MA><1>': 392.7424603174595,
  'saavedra': 3.875,
  'el': 377.54603174602966,
  'ingenioso': 1.801190476190476,
  'hidalgo': 4.820634920634922,
  'don': 136.49801587301585,
  'quijote': 127.29960317460338,
  'la': 432.0269841269835,
  'mancha': 35.31230158730156,
  'al': 89.30515873015885,
  'duque': 19.048809523809513,
  'bejar': 0.9166666666666666,
  '<MA><1><AC><3>': 10.751190476190475,
  'marques': 1.3527777777777779,
  '<MA><1><MA><2><MA><3><MA><4

In [18]:
! /usr/local/bin/python3.12 -m pip install scipy


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [19]:
from scipy import sparse

filas = []
columnas = []
datos = []

for token_i, vecinos in W.items():
    for token_j, distancia in vecinos.items():
        i = quijote_corpus_diccionario_token_indices[token_i]
        j = quijote_corpus_diccionario_token_indices[token_j]
        filas.append(i)
        columnas.append(j)
        datos.append(distancia)

# Máxima longitud
V = max(max(filas), max(columnas)) + 1

Ws = sparse.csr_matrix(
    (datos, (filas, columnas)),
    shape=(V, V)
)

Ws.shape

(18356, 18356)

In [20]:
! /usr/local/bin/python3.12 -m pip install scikit-learn


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [21]:
from sklearn import decomposition

m = 8 # Número de características espaciales que se capturan de las distancias entre palabras

# Descomposición matricial por los m-autovalores principales
svd = decomposition.TruncatedSVD(m, random_state=123)

E = svd.fit_transform(Ws) # Alternativamente se puede usar PCA, pero cambia el significado

E # 18,356 x 8

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 2.11530770e+00,  2.44293592e+00, -1.65796671e+00, ...,
         1.09179131e+00, -2.87360573e+00, -4.41275592e-01],
       [ 1.75221437e+03,  6.99432157e+01, -1.92204540e+02, ...,
        -6.79080803e+01,  4.55132291e+01, -2.19015233e+01],
       ...,
       [ 2.18551264e-01, -7.52106495e-02, -3.11090823e-02, ...,
        -1.04660600e-01, -1.25323120e-01, -2.63489273e-02],
       [ 1.67354807e-01, -2.26518336e-02, -3.66569550e-02, ...,
        -4.90984214e-02,  2.03171900e-02,  2.78091926e-02],
       [ 2.73885884e-01, -1.58390673e-01, -1.08323722e-01, ...,
        -3.07166055e-02, -1.14767407e-02, -1.60681437e-01]])

> Correlaciones entre tokens

W - 18,356 x 18,356

i / j A B C ... XXXX
A     0 1.5 0.5 ... 0.3
B     0.8 0 ... 0
C     0   0 ... 0
...
XXXXX

In [22]:
5 * 18356

91780

In [23]:
# 5 palabras (W-18,356, W-18,356, W-18,356, W-18,356, W-18,356) - 91780 características
# 5 palabras (E-8, E-8, E-8, E-8, E-8) - 40 características

# 80 palabras (E-8, E-8, E-8, E-8, E-8, ...) - 640 características

## Entrenar un modelo con embeddings

1. Fijan las sentencias a una cantidad máxima y mínima de tokens y si hacen falta rellenan con 0 y sino truncan
2. Sentencias o textos se transforman en una matriz X de embeddings (flatten / promedio / Conv1D)
3. Predicen Y <- X

In [32]:
quijote_corpus_diccionario_token_indices["quijote"], quijote_corpus_diccionario_token_indices["hidalgo"]

(10, 8)

In [35]:
W["quijote"]["hidalgo"]

0.2361111111111111

In [37]:
E[quijote_corpus_diccionario_token_indices["quijote"]], E[quijote_corpus_diccionario_token_indices["hidalgo"]]

(array([ 474.89469185,  408.35528953, -254.3532421 , -212.63688643,
         -77.0593647 ,  177.06942154, -313.54170902,  -34.14295199]),
 array([11.2170432 ,  4.02787525, -1.26509297, -0.28355168,  1.31756483,
        -1.36825003, -1.59348598,  0.83364942]))

## Similitud coseno

$$
cos(\theta) = \frac{\overline{u} \cdot \overline{v}}{\|\overline{u}\| \|\overline{v}\|}
$$

In [41]:
import numpy

u = E[quijote_corpus_diccionario_token_indices["quijote"]]
v = E[quijote_corpus_diccionario_token_indices["hidalgo"]]

numpy.dot(u, v) / (numpy.linalg.norm(u) * numpy.linalg.norm(v))

np.float64(0.7627204764913984)

In [40]:
u = E[quijote_corpus_diccionario_token_indices["quijote"]]
v = E[quijote_corpus_diccionario_token_indices["caballero"]]

numpy.dot(u, v) / (numpy.linalg.norm(u) * numpy.linalg.norm(v))

np.float64(0.7731223623753594)

In [42]:
u = E[quijote_corpus_diccionario_token_indices["quijote"]]
v = E[quijote_corpus_diccionario_token_indices["cantar"]]

numpy.dot(u, v) / (numpy.linalg.norm(u) * numpy.linalg.norm(v))

np.float64(0.4597773047997512)

In [43]:
u = E[quijote_corpus_diccionario_token_indices["quijote"]]
v = E[quijote_corpus_diccionario_token_indices["sombra"]]

numpy.dot(u, v) / (numpy.linalg.norm(u) * numpy.linalg.norm(v))

np.float64(0.4492532113711056)

In [44]:
u = E[quijote_corpus_diccionario_token_indices["sombra"]]
v = E[quijote_corpus_diccionario_token_indices["quijote"]]

numpy.dot(u, v) / (numpy.linalg.norm(u) * numpy.linalg.norm(v))

np.float64(0.4492532113711056)

In [24]:
E

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 2.11530770e+00,  2.44293592e+00, -1.65796671e+00, ...,
         1.09179131e+00, -2.87360573e+00, -4.41275592e-01],
       [ 1.75221437e+03,  6.99432157e+01, -1.92204540e+02, ...,
        -6.79080803e+01,  4.55132291e+01, -2.19015233e+01],
       ...,
       [ 2.18551264e-01, -7.52106495e-02, -3.11090823e-02, ...,
        -1.04660600e-01, -1.25323120e-01, -2.63489273e-02],
       [ 1.67354807e-01, -2.26518336e-02, -3.66569550e-02, ...,
        -4.90984214e-02,  2.03171900e-02,  2.78091926e-02],
       [ 2.73885884e-01, -1.58390673e-01, -1.08323722e-01, ...,
        -3.07166055e-02, -1.14767407e-02, -1.60681437e-01]])

In [51]:
sentencia1_texto = "Me siento desdeñado por la gloriosa vida inútil"
sentencia2_texto = "Me siento desdeñado por el inútil dolor de la vida"
sentencia3_texto = "Dulcinea del Toboso es la más hermosa mujer del mundo"

sentencia1_vector = tokenizar(sentencia1_texto)
sentencia2_vector = tokenizar(sentencia2_texto)
sentencia3_vector = tokenizar(sentencia3_texto)

print(sentencia1_vector)
print(sentencia2_vector)
print(sentencia3_vector)

['me', '<MA><1>', 'siento', 'desdenado', '<TI><7>', 'por', 'la', 'gloriosa', 'vida', 'inutil', '<AC><4>']
['me', '<MA><1>', 'siento', 'desdenado', '<TI><7>', 'por', 'el', 'inutil', '<AC><4>', 'dolor', 'de', 'la', 'vida']
['dulcinea', '<MA><1>', 'del', 'toboso', '<MA><1>', 'es', 'la', 'mas', '<AC><3>', 'hermosa', 'mujer', 'del', 'mundo']


> Padding: Rellenar la sentencia a un número comprensible de tokens a procesar (30 tokens)

In [57]:
sentencia1_padding = (sentencia1_vector + ["<nulo>"] * 30)[:30]
sentencia2_padding = (sentencia2_vector + ["<nulo>"] * 30)[:30]
sentencia3_padding = (sentencia3_vector + ["<nulo>"] * 30)[:30]

print(sentencia1_padding)
print(sentencia2_padding)
print(sentencia3_padding)

['me', '<MA><1>', 'siento', 'desdenado', '<TI><7>', 'por', 'la', 'gloriosa', 'vida', 'inutil', '<AC><4>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>']
['me', '<MA><1>', 'siento', 'desdenado', '<TI><7>', 'por', 'el', 'inutil', '<AC><4>', 'dolor', 'de', 'la', 'vida', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>']
['dulcinea', '<MA><1>', 'del', 'toboso', '<MA><1>', 'es', 'la', 'mas', '<AC><3>', 'hermosa', 'mujer', 'del', 'mundo', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>']


In [79]:
import pandas

import seaborn

def get_embedding(token):
    if not token in quijote_corpus_diccionario_token_indices:
        return numpy.zeros(m)
    return E[quijote_corpus_diccionario_token_indices[token]]

# Matriz embedding (30x8) - CNN
sentencia1_embedding = numpy.array([get_embedding(token) for token in sentencia1_padding])
sentencia2_embedding = numpy.array([get_embedding(token) for token in sentencia2_padding])
sentencia3_embedding = numpy.array([get_embedding(token) for token in sentencia3_padding])
# Matriz embedding aplanada (30*8 = 240) - NN
sentencia1_embedding_flatten = sentencia1_embedding.flatten()
sentencia2_embedding_flatten = sentencia2_embedding.flatten()
sentencia3_embedding_flatten = sentencia3_embedding.flatten()
# Matriz embedding promedio (8) - NN
sentencia1_embedding_mean = sentencia1_embedding.mean(axis=0)
sentencia2_embedding_mean = sentencia2_embedding.mean(axis=0)
sentencia3_embedding_mean = sentencia3_embedding.mean(axis=0)

# seaborn.heatmap(sentencia1_embedding, cmap="Blues")
pandas.DataFrame([
    sentencia1_embedding_mean,
    sentencia2_embedding_mean,
    sentencia3_embedding_mean,
])

,0,1,2,3,4,5,6,7
0,151.015503,3.958359,-5.284773,16.392855,2.309880,-7.659377,-5.587916,9.635637
1,291.106226,23.621508,28.880651,21.380758,6.751936,-10.495424,-10.172573,20.998027
2,258.477140,9.495925,-20.961613,42.553527,35.943287,15.925191,-7.134048,8.845339


In [83]:
quijote_narracion_dialogo = pandas.read_csv("../conjuntos/quijote_narracion_dialogo.csv")

quijote_narracion_dialogo["X_tokens"] = quijote_narracion_dialogo["X"].map(tokenizar)

quijote_narracion_dialogo

: 58 6
; 59 5
; 59 6
. 46 5
. 46 5
. 46 10
. 46 7
; 59 11
. 46 10
. 46 8
: 58 6
; 59 8
« 171 1
» 187 7
. 46 6
. 46 7
¡ 161 1
! 33 6
. 46 16
- 45 7
. 46 7
. 46 8
; 59 4
: 58 8
: 58 6
; 59 9
. 46 7
; 59 6
; 59 6
. 46 7
; 59 8
. 46 6
. 46 6
- 45 9
; 59 11
. 46 8
. 46 9
. 46 10
. 46 4
. 46 6
. 46 9
; 59 6
: 58 5
. 46 9
; 59 7
; 59 9
. 46 5
¡ 161 1
. 46 5
. 46 5
¡ 161 1
! 33 6
. 46 8
. 46 4
¡ 161 1
! 33 12
- 45 7
: 58 5
- 45 7
. 46 10
¡ 161 1
! 33 4
; 59 5
. 46 9
; 59 6
. 46 8
; 59 8
: 58 9
; 59 7
. 46 8
: 58 5
. 46 5
: 58 6
. 46 6
« 171 1
» 187 11
. 46 6
. 46 8
« 171 1
» 187 7
. 46 8
: 58 6
; 59 6
. 46 7
. 46 13
. 46 8
. 46 6
. 46 7
. 46 13
¡ 161 1
! 33 10
. 46 8
. 46 8
. 46 9
; 59 7
: 58 5
. 46 5
: 58 5
; 59 7
. 46 6
. 46 4
- 45 8
: 58 7
. 46 11
. 46 8
. 46 5
; 59 14
. 46 6
. 46 7
; 59 7
: 58 5
. 46 7
. 46 7
: 58 8
. 46 11
- 45 8
: 58 9
. 46 5
; 59 13
: 58 6
: 58 9
. 46 12
. 46 8
. 46 14
- 45 8
. 46 7
. 46 8
: 58 11
« 171 1
» 187 6
. 46 7
- 45 9
. 46 6
- 45 7
; 59 6
. 46 4
- 45 8
. 46 7
.

,X,Y1,Y2,X_tokens
0,Nadie sabe lo que está por venir: de aquí a ma...,0,0,"[nadie, <MA><1>, sabe, lo, que, esta, <AC><5>,..."
1,La de la caballería andante,0,1,"[la, <MA><1>, de, la, caballeria, <AC><10>, an..."
2,¿Ahí está el señor Florismarte?,0,1,"[ahi, <IA><1><MA><2><AC><5>, esta, <AC><5>, el..."
3,"Don Quijote mi amo, según he oído decir en est...",1,0,"[don, <MA><1>, quijote, <MA><1>, mi, amo, <PC>..."
4,¿Cómo si es así?,0,1,"[como, <IA><1><MA><2><AC><4>, si, es, asi, <AC..."
...,...,...,...,...
3702,En las suyas sintieron los que escuchado la ha...,0,0,"[en, <MA><1>, las, suyas, sintieron, los, que,..."
3703,No entiendo eso de logicuos,0,1,"[no, <MA><1>, entiendo, eso, de, logicuos]"
3704,"Quísele antecoger delante de mí y traérosle, p...",0,1,"[quisele, <MA><1><AC><4>, antecoger, delante, ..."
3705,«Nunca fuera caballero de damas tan bien servi...,0,1,"[nunca, <ND><MA><2>, fuera, caballero, de, dam..."


In [89]:
quijote_narracion_dialogo["X_tokens"].map(lambda tokens: len(tokens)).quantile(0.9)

np.float64(86.0)

In [90]:
PADDING_SIZE = 80

def padding(tokens):
    return (tokens + ["<nulo>"] * 80)[:80]

quijote_narracion_dialogo["X_sentencias"] = quijote_narracion_dialogo["X_tokens"].map(padding)

quijote_narracion_dialogo

,X,Y1,Y2,X_tokens,X_sentencias
0,Nadie sabe lo que está por venir: de aquí a ma...,0,0,"[nadie, <MA><1>, sabe, lo, que, esta, <AC><5>,...","[nadie, <MA><1>, sabe, lo, que, esta, <AC><5>,..."
1,La de la caballería andante,0,1,"[la, <MA><1>, de, la, caballeria, <AC><10>, an...","[la, <MA><1>, de, la, caballeria, <AC><10>, an..."
2,¿Ahí está el señor Florismarte?,0,1,"[ahi, <IA><1><MA><2><AC><5>, esta, <AC><5>, el...","[ahi, <IA><1><MA><2><AC><5>, esta, <AC><5>, el..."
3,"Don Quijote mi amo, según he oído decir en est...",1,0,"[don, <MA><1>, quijote, <MA><1>, mi, amo, <PC>...","[don, <MA><1>, quijote, <MA><1>, mi, amo, <PC>..."
4,¿Cómo si es así?,0,1,"[como, <IA><1><MA><2><AC><4>, si, es, asi, <AC...","[como, <IA><1><MA><2><AC><4>, si, es, asi, <AC..."
...,...,...,...,...,...
3702,En las suyas sintieron los que escuchado la ha...,0,0,"[en, <MA><1>, las, suyas, sintieron, los, que,...","[en, <MA><1>, las, suyas, sintieron, los, que,..."
3703,No entiendo eso de logicuos,0,1,"[no, <MA><1>, entiendo, eso, de, logicuos]","[no, <MA><1>, entiendo, eso, de, logicuos, <nu..."
3704,"Quísele antecoger delante de mí y traérosle, p...",0,1,"[quisele, <MA><1><AC><4>, antecoger, delante, ...","[quisele, <MA><1><AC><4>, antecoger, delante, ..."
3705,«Nunca fuera caballero de damas tan bien servi...,0,1,"[nunca, <ND><MA><2>, fuera, caballero, de, dam...","[nunca, <ND><MA><2>, fuera, caballero, de, dam..."


In [91]:
def get_embedding_full(sentence):
    return numpy.array([get_embedding(token) for token in sentence])

def get_embedding_flatten(sentence):
    return get_embedding_full(sentence).flatten()

def get_embedding_mean(sentence):
    return get_embedding_full(sentence).mean(axis=0)

quijote_narracion_dialogo["X_mean"] = quijote_narracion_dialogo["X_sentencias"].apply(get_embedding_mean)

quijote_narracion_dialogo

,X,Y1,Y2,X_tokens,X_sentencias,X_mean
0,Nadie sabe lo que está por venir: de aquí a ma...,0,0,"[nadie, <MA><1>, sabe, lo, que, esta, <AC><5>,...","[nadie, <MA><1>, sabe, lo, que, esta, <AC><5>,...","[572.8232597845077, -0.7477138753880654, -14.1..."
1,La de la caballería andante,0,1,"[la, <MA><1>, de, la, caballeria, <AC><10>, an...","[la, <MA><1>, de, la, caballeria, <AC><10>, an...","[95.41709303355705, 10.5472752388216, 10.15833..."
2,¿Ahí está el señor Florismarte?,0,1,"[ahi, <IA><1><MA><2><AC><5>, esta, <AC><5>, el...","[ahi, <IA><1><MA><2><AC><5>, esta, <AC><5>, el...","[41.53185826514385, -0.9425919644288532, -2.56..."
3,"Don Quijote mi amo, según he oído decir en est...",1,0,"[don, <MA><1>, quijote, <MA><1>, mi, amo, <PC>...","[don, <MA><1>, quijote, <MA><1>, mi, amo, <PC>...","[317.1775653541802, -0.44958698140825104, -27...."
4,¿Cómo si es así?,0,1,"[como, <IA><1><MA><2><AC><4>, si, es, asi, <AC...","[como, <IA><1><MA><2><AC><4>, si, es, asi, <AC...","[15.64501488713285, -2.75794693518652, -0.7393..."
...,...,...,...,...,...,...
3702,En las suyas sintieron los que escuchado la ha...,0,0,"[en, <MA><1>, las, suyas, sintieron, los, que,...","[en, <MA><1>, las, suyas, sintieron, los, que,...","[368.7411147316485, 5.667445435374359, -14.067..."
3703,No entiendo eso de logicuos,0,1,"[no, <MA><1>, entiendo, eso, de, logicuos]","[no, <MA><1>, entiendo, eso, de, logicuos, <nu...","[68.00084074281433, 4.2720493331603935, 10.557..."
3704,"Quísele antecoger delante de mí y traérosle, p...",0,1,"[quisele, <MA><1><AC><4>, antecoger, delante, ...","[quisele, <MA><1><AC><4>, antecoger, delante, ...","[371.42946820376244, -10.59073207243071, -0.27..."
3705,«Nunca fuera caballero de damas tan bien servi...,0,1,"[nunca, <ND><MA><2>, fuera, caballero, de, dam...","[nunca, <ND><MA><2>, fuera, caballero, de, dam...","[170.7088670558848, 39.73073683670488, 4.52226..."


In [104]:
X = numpy.stack(quijote_narracion_dialogo["X_mean"].values)

X

array([[ 5.72823260e+02, -7.47713875e-01, -1.41607518e+01, ...,
         1.03482053e+01, -2.74741434e+00, -1.68462424e+00],
       [ 9.54170930e+01,  1.05472752e+01,  1.01583382e+01, ...,
        -3.75723826e+00, -3.30431198e+00,  6.60339362e+00],
       [ 4.15318583e+01, -9.42591964e-01, -2.56201584e+00, ...,
        -3.77691013e+00, -4.81048085e+00,  6.37991541e+00],
       ...,
       [ 3.71429468e+02, -1.05907321e+01, -2.75004770e-01, ...,
         8.28504555e+00, -1.28626487e+01, -1.27930371e+01],
       [ 1.70708867e+02,  3.97307368e+01,  4.52226712e+00, ...,
         1.34383357e+00,  4.62433190e+00,  7.94965909e+00],
       [ 2.67409522e+02,  7.26583435e+00, -1.80380060e+01, ...,
        -7.00683773e+00,  4.05177144e+00,  1.34721731e+01]])

In [111]:
Y1 = quijote_narracion_dialogo["Y1"]
Y2 = quijote_narracion_dialogo["Y2"]

Y = pandas.DataFrame([Y1, Y2]).T

Y

,Y1,Y2
0,0,0
1,0,1
2,0,1
3,1,0
4,0,1
...,...,...
3702,0,0
3703,0,1
3704,0,1
3705,0,1


In [107]:
import keras

In [109]:
modelo = keras.Sequential([
    keras.Input(shape=(8,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(2, activation="sigmoid"),
])

modelo.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

modelo.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,410 (5.51 KB)

 Trainable params: 1,410 (5.51 KB)

 Non-trainable params: 0 (0.00 B)

In [113]:
modelo.fit(X, Y, epochs=200)

Epoch 1/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8834 - loss: 0.4178
Epoch 2/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 601us/step - accuracy: 0.8697 - loss: 0.4256
Epoch 3/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 762us/step - accuracy: 0.8627 - loss: 0.4411
Epoch 4/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 584us/step - accuracy: 0.8757 - loss: 0.4369
Epoch 5/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 542us/step - accuracy: 0.8786 - loss: 0.4301
Epoch 6/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 549us/step - accuracy: 0.8793 - loss: 0.4263
Epoch 7/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 660us/step - accuracy: 0.8692 - loss: 0.4331
Epoch 8/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 638us/step - accuracy: 0.8838 - loss: 0.4274
Epoch 9/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 742us/step - accuracy: 0.8822 - loss: 0.4172
Epoch 10/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 703us/step - accuracy: 0.8683 - loss: 0.4399
Epoch 11/200
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 706us/step - accuracy: 0.8831 - loss: 0.4334
Epoch 12/200
116/116 

In [117]:
sentencia1_padding = (sentencia1_vector + ["<nulo>"] * 80)[:80]
sentencia2_padding = (sentencia2_vector + ["<nulo>"] * 80)[:80]
sentencia3_padding = (sentencia3_vector + ["<nulo>"] * 80)[:80]

print(sentencia1_padding)
print(sentencia2_padding)
print(sentencia3_padding)

['me', '<MA><1>', 'siento', 'desdenado', '<TI><7>', 'por', 'la', 'gloriosa', 'vida', 'inutil', '<AC><4>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>']
['me', '<MA><1>', 'siento', 'desdenado', '<TI><7>', 'por', 'el', 'inutil', '<AC><4>', 'dolor', 'de', 'la', 'vida', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>', '<nulo>',

In [118]:
import pandas

import seaborn

def get_embedding(token):
    if not token in quijote_corpus_diccionario_token_indices:
        return numpy.zeros(m)
    return E[quijote_corpus_diccionario_token_indices[token]]

# Matriz embedding (80x8) - CNN
sentencia1_embedding = numpy.array([get_embedding(token) for token in sentencia1_padding])
sentencia2_embedding = numpy.array([get_embedding(token) for token in sentencia2_padding])
sentencia3_embedding = numpy.array([get_embedding(token) for token in sentencia3_padding])
# Matriz embedding aplanada (80*8 = 240) - NN
sentencia1_embedding_flatten = sentencia1_embedding.flatten()
sentencia2_embedding_flatten = sentencia2_embedding.flatten()
sentencia3_embedding_flatten = sentencia3_embedding.flatten()
# Matriz embedding promedio (8) - NN
sentencia1_embedding_mean = sentencia1_embedding.mean(axis=0)
sentencia2_embedding_mean = sentencia2_embedding.mean(axis=0)
sentencia3_embedding_mean = sentencia3_embedding.mean(axis=0)

# seaborn.heatmap(sentencia1_embedding, cmap="Blues")
pandas.DataFrame([
    sentencia1_embedding_mean,
    sentencia2_embedding_mean,
    sentencia3_embedding_mean,
])

,0,1,2,3,4,5,6,7
0,56.630814,1.484385,-1.981790,6.147321,0.866205,-2.872266,-2.095468,3.613364
1,109.164835,8.858065,10.830244,8.017784,2.531976,-3.935784,-3.814715,7.874260
2,96.928927,3.560972,-7.860605,15.957573,13.478732,5.971947,-2.675268,3.317002


In [120]:
# sentencia1_texto = "Me siento desdeñado por la gloriosa vida inútil"
# sentencia2_texto = "Me siento desdeñado por el inútil dolor de la vida"
# sentencia3_texto = "Dulcinea del Toboso es la más hermosa mujer del mundo"

modelo.predict(pandas.DataFrame([
    sentencia1_embedding_mean,
    sentencia2_embedding_mean,
    sentencia3_embedding_mean,
]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step


array([[0.0172636 , 0.91705924],
       [0.03884706, 0.6633919 ],
       [0.05108311, 0.491882  ]], dtype=float32)